# 1. Problem Statement & Goals 🎯
___
## Problem Statement
Ebuss, a growing e-commerce company with a significant market share in categories like household essentials, personal care, and electronics, aims to scale rapidly and compete with market leaders like Amazon and Flipkart.

To achieve this, Ebuss needs to leverage its vast data on user reviews and ratings. As a Senior Machine Learning Engineer, the core challenge is to build a sentiment-based product recommendation system. This system must not only recommend products based on user behaviors (ratings) but also refine those recommendations by analyzing the sentiment of the textual reviews associated with those products. The ultimate objective is to enhance the user experience by suggesting products that users are most likely to purchase and feel positive about.

## Goals
The project is divided into four main objectives to achieve the problem statement:

### 1. Data Sourcing and Sentiment Analysis

#### Objective: 
* Build a Machine Learning model to classify user reviews as Positive or Negative.

#### Key Tasks:

* Perform Exploratory Data Analysis (EDA), data cleaning, and text preprocessing.

* Extract features using techniques like Bag-of-Words, TF-IDF, or Word Embeddings.

* Train and evaluate at least three of the following classification models: Logistic Regression, Random Forest, XGBoost, or Naive Bayes.

* Select the best-performing model to predict user sentiment.

### 2. Building a Recommendation System

#### Objective: 
* Identify the most effective recommendation technique for the dataset.

#### Key Tasks:

* Develop both User-based and Item-based collaborative filtering recommendation systems.

* Analyze and compare their performance to select the best-suited system.

* Generate an initial list of 20 recommended products for a specific user based on their historical ratings.

### 3. Improving Recommendations using Sentiment Analysis

#### Objective: 
* Create a hybrid "Sentiment-Based Recommendation System."

#### Key Tasks:

* Integrate the chosen Sentiment Analysis model with the Recommendation System.

* Take the top 20 products recommended by the collaborative filtering system.

* Filter and rank these products based on their predicted sentiment scores.

* Output the final top 5 products that have the highest positive sentiment.

### 4. Deployment

#### Objective: 
* Make the solution accessible via a web interface.

#### Key Tasks:

* Build a web application using the Flask framework.

* Create a User Interface (UI) that accepts a username and displays the top 5 recommended products.

* Deploy the end-to-end application (Model + API + UI) on a cloud platform like Heroku.

In [1]:
# Importing Required Libraries
import random
from pathlib import Path
import os

# Data Science Libraries
import pandas as pd

# Notebook Setup
from notebook_setup import NotebookInitializer
# Pass the path of the current file to the initializer
initializer = NotebookInitializer(Path(os.getcwd()).resolve())
initializer.setup_environment()

# Local Utils
from src.utils.data_loader import DataLoader
from src.utils.text_processing import TextProcessor, StandardizeNameData
from src.utils.imputation import CategoryImputer
from src.utils.data_cleaner import DataCleaner
from src.utils.data_analyzer import DataAnalyzer

ROOT_DIR already set to: D:\Projects\PRS
Original working directory: d:\Projects\PRS\notebooks
Current working directory changed to the project root: D:\Projects\PRS

--- Directory Structure Setup ---
📂 Root Directory: D:\Projects\PRS
📁 Data Directory: D:\Projects\PRS\data
📥 Raw Data Directory: D:\Projects\PRS\data\raw
📤 Processed Data Directory: D:\Projects\PRS\data\processed
⚙️ Config Manager: ConfigManager(config_path=src/config.json, model_path=models/)
Importing ConfigManager from parent directory


# 2. Setup and Data Loading ⚙️

In [2]:
# Assign the returned pandas DataFrame from the 'load_data' function to the variable 'df'.
# The function is passed a file path, which is constructed in a platform-independent way
# using the pathlib library. The '/' operator joins the 'ROW_DATA_DIR' Path object
# with the filename "dataset.csv".
raw_data_path = initializer.raw_data_dir / "dataset.csv"
df_raw = DataLoader.load_csv_data(raw_data_path)

Loading data from D:\Projects\PRS\data\raw\dataset.csv
Data successfully loaded. Shape: (30000, 15)


## Initial Inspection

### 📘 Dataset Attribute Description

| Attribute              | Description                                                                                                                           |
|------------------------|---------------------------------------------------------------------------------------------------------------------------------------|
| **id**                 | Unique identity number to identify each review given by a user to a particular product in the dataset.                                |
| **brand**              | Name of the brand of the product being reviewed.                                                                                      |
| **categories**         | Category of the product (e.g., household essentials, books, personal care, medicines, cosmetics, beauty products, appliances, etc.). |
| **manufacturer**       | Name of the manufacturer of the product.                                                                                              |
| **name**               | Name of the product for which the review or rating was added.                                                                          |
| **reviews_date**       | Date on which the review was added by the user.                                                                                       |
| **reviews_didPurchase**| Indicates whether the user purchased the product.                                                                                     |
| **reviews_doRecommend**| Indicates whether the user recommends the product.                                                                                    |
| **reviews_rating**     | Rating given by the user to the product.                                                                                              |
| **reviews_text**       | Text of the review written by the user.                                                                                               |
| **reviews_title**      | Title of the review provided by the user.                                                                                             |
| **reviews_userCity**   | City where the user resides.                                                                                                          |
| **reviews_userProvince**| Province/state where the user resides.                                                                                               |
| **reviews_username**   | Unique identifier for the individual user in the dataset.                                                                             |
| **user_sentiment**     | Overall sentiment of the user for the product (Positive or Negative).                                                                  |


In [3]:
# This will print the first 5 rows of the DataFrame to the console
display(df_raw.head())

,id,brand,categories,manufacturer,name,reviews_date,reviews_didPurchase,reviews_doRecommend,reviews_rating,reviews_text,reviews_title,reviews_userCity,reviews_userProvince,reviews_username,user_sentiment
0,AV13O1A8GV-KLJ3akUyj,Universal Music,"Movies, Music & Books,Music,R&b,Movies & TV,Mo...",Universal Music Group / Cash Money,Pink Friday: Roman Reloaded Re-Up (w/dvd),2012-11-30T06:21:45.000Z,NaN,NaN,5,i love this album. it's very good. more to the...,Just Awesome,Los Angeles,NaN,joshua,Positive
1,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor. This review was collected as part...,Good,NaN,NaN,dorothy w,Positive
2,AV14LG0R-jtxr-f38QfS,Lundberg,"Food,Packaged Foods,Snacks,Crackers,Snacks, Co...",Lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,2017-07-09T00:00:00.000Z,True,NaN,5,Good flavor.,Good,NaN,NaN,dorothy w,Positive
3,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-01-06T00:00:00.000Z,False,False,1,I read through the reviews on here before look...,Disappointed,NaN,NaN,rebecca,Negative
4,AV16khLE-jtxr-f38VFn,K-Y,"Personal Care,Medicine Cabinet,Lubricant/Sperm...",K-Y,K-Y Love Sensuality Pleasure Gel,2016-12-21T00:00:00.000Z,False,False,1,My husband bought this gel for us. The gel cau...,Irritation,NaN,NaN,walker557,Negative


In [4]:
# This will print the last 5 rows of the DataFrame to the console
display(df_raw.tail())

,id,brand,categories,manufacturer,name,reviews_date,reviews_didPurchase,reviews_doRecommend,reviews_rating,reviews_text,reviews_title,reviews_userCity,reviews_userProvince,reviews_username,user_sentiment
29995,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-23T00:00:00.000Z,False,True,5,I got this conditioner with Influenster to try...,Softness!!,NaN,NaN,laurasnchz,Positive
29996,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-27T00:00:00.000Z,False,True,5,"I love it , I received this for review purpose...",I love it,NaN,NaN,scarlepadilla,Positive
29997,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-21T00:00:00.000Z,False,True,5,First of all I love the smell of this product....,Hair is so smooth after use,NaN,NaN,liviasuexo,Positive
29998,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-11T00:00:00.000Z,False,True,5,I received this through Influenster and will n...,Perfect for my oily hair!,NaN,NaN,ktreed95,Positive
29999,AVpfW8y_LJeJML437ySW,L'oreal Paris,"Beauty,Hair Care,Shampoo & Conditioner,Holiday...",L'oreal Paris,L'or233al Paris Elvive Extraordinary Clay Reba...,2017-01-19T00:00:00.000Z,False,True,5,I received this product complimentary from inf...,Conditioned into healthy,NaN,NaN,kcoopxoxo,Positive


In [5]:
# Print dimensionality
rows, cols = df_raw.shape
print(f"✅ Dataframe Shape: {rows:,} rows x {cols} columns")

✅ Dataframe Shape: 30,000 rows x 15 columns


In [6]:
# Print schema and non-null counts
print("\n📊 Dataframe Info:")
# verbose=True ensures full info is printed even for large DataFrames
df_raw.info(verbose=True, show_counts=True)


📊 Dataframe Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    30000 non-null  object
 1   brand                 30000 non-null  object
 2   categories            30000 non-null  object
 3   manufacturer          29859 non-null  object
 4   name                  30000 non-null  object
 5   reviews_date          29954 non-null  object
 6   reviews_didPurchase   15932 non-null  object
 7   reviews_doRecommend   27430 non-null  object
 8   reviews_rating        30000 non-null  int64 
 9   reviews_text          30000 non-null  object
 10  reviews_title         29810 non-null  object
 11  reviews_userCity      1929 non-null   object
 12  reviews_userProvince  170 non-null    object
 13  reviews_username      29937 non-null  object
 14  user_sentiment        29999 non-null  object
dtypes: int64(1), obje

In [7]:
missing_cols = DataAnalyzer.analyze_missing_data(df_raw)
initializer.config.set_config("missing_cols", missing_cols)

📊 Missing Values Report
Missing Values and Percentages (Filtered by reporting threshold):
|                      |   Missing Count |   Missing Percentage (%) |   Unique Values Count |
|:---------------------|----------------:|-------------------------:|----------------------:|
| reviews_userProvince |           29830 |                    99.43 |                    42 |
| reviews_userCity     |           28071 |                    93.57 |                   977 |
| reviews_didPurchase  |           14068 |                    46.89 |                     2 |
| reviews_doRecommend  |            2570 |                     8.57 |                     2 |
| reviews_title        |             190 |                     0.63 |                 18535 |
| manufacturer         |             141 |                     0.47 |                   227 |
| reviews_username     |              63 |                     0.21 |                 24914 |
| reviews_date         |              46 |                     0

#### Key Points Summary

##### `id` Column

* Represents only a **unique row identifier** → no statistical or predictive value.
* Not useful for classification, regression, sentiment analysis, or recommendations.
* Keeping it may **add noise** or encourage overfitting in certain models.
* Dropping helps **reduce memory footprint** and simplifies the feature space.

##### `reviews_didPurchase` Column

* Contains **~46% missing values**, making the feature unreliable.
* High missingness makes **imputation inaccurate** and could bias the model.
* Core ML tasks (sentiment analysis / recommendation) **do not depend** on this flag.
* Offers limited improvement to prediction accuracy given the **incomplete data distribution**.
* Dropping avoids unnecessary preprocessing complexity with minimal model impact.


In [8]:
# As per above analysis check prev cell markdown
invalid_cols = ['id', 'reviews_didPurchase']
initializer.config.set_config("invalid_cols", invalid_cols)

In [9]:
DataAnalyzer.analyze_value_distribution(df_raw, 'reviews_doRecommend')

🔍 Distribution Analysis for 'reviews_doRecommend'
Found 3 unique values (including NaN if dropna is False).
Value Counts Distribution:
| reviews_doRecommend   | Count   | Percentage (%)   |
|:----------------------|:--------|:-----------------|
| 1                     | 25880   | 86.27            |
| nan                   | 2570    | 8.57             |
| 0                     | 1550    | 5.17             |


#### 📊 Data Analysis

Based on the analysis of missing values in the dataset, we have observed the following:

##### 1. **Critical Missing Data (Columns to Remove)**
The following columns have an extremely high percentage of missing values (over 90%). They contain very little actual data and will not be useful for our analysis or modeling. We should **drop** these columns:
*   **`reviews_userProvince`**: ~99.4% missing.
*   **`reviews_userCity`**: ~93.6% missing.

##### 2. **Minimal Missing Data**
The remaining columns have a low percentage of missing values and can be handled by either dropping the specific rows or imputing the values:
*   **`reviews_title`**, **`manufacturer`**, **`reviews_username`**, **`reviews_date`**, **`user_sentiment`**: All have less than 1% missing data.

# 3. Data Cleaning and Preprocessing 🛠️

In [10]:
df_cleaned = DataCleaner.drop_specified_columns(df_raw, missing_value_columns=missing_cols, invalid_value_columns=invalid_cols)

🗑️ Column Dropping Utility
Dropped columns: ['reviews_userProvince', 'reviews_userCity', 'reviews_didPurchase', 'id']
Initial DataFrame shape: (30000, 15)
Final DataFrame shape: (30000, 11)

Cleaned Data Preview:
|    | brand           | categories                                                                                                                                                     | manufacturer                       | name                                       | reviews_date             |   reviews_doRecommend |   reviews_rating | reviews_text                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [11]:
df_cleaned.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   brand                30000 non-null  object
 1   categories           30000 non-null  object
 2   manufacturer         29859 non-null  object
 3   name                 30000 non-null  object
 4   reviews_date         29954 non-null  object
 5   reviews_doRecommend  27430 non-null  object
 6   reviews_rating       30000 non-null  int64 
 7   reviews_text         30000 non-null  object
 8   reviews_title        29810 non-null  object
 9   reviews_username     29937 non-null  object
 10  user_sentiment       29999 non-null  object
dtypes: int64(1), object(10)
memory usage: 2.5+ MB


In [12]:
# Processing Pipeline
print("Starting transformation pipeline...")

print("\nProcessing Name column text data...")
text_processor = TextProcessor()
print("Name text processing init completed.")
cleaned_names = text_processor.process_batch(df_cleaned["name"].tolist())
print("Text processing on name column completed.")


print("\nProcessing Brand column text data...")
brand_std = StandardizeNameData(target_column="brand")
print("Brand standardization init completed.")
brand_clean = brand_std.transform_series(df_cleaned["brand"])
print("Brand standardization completed.")


print("\nProcessing Manufacturer column text data...")
mfr_std = StandardizeNameData(target_column="manufacturer")
print("Manufacturer standardization init completed.")   
mfr_clean = mfr_std.transform_series(df_cleaned["manufacturer"])
print("Manufacturer standardization completed.")


# Assign results safely
df_cleaned = df_cleaned.assign(
    brand_clean=brand_clean,
    manufacturer_clean=mfr_clean,
    name_clean=cleaned_names
)
# Display results
print("Pipeline completed.")
print("📊 Displaying first 5 rows of the cleaned data...")
display(df_cleaned[["brand", "brand_clean", "manufacturer", "manufacturer_clean", "name", "name_clean"]].head())

Starting transformation pipeline...

Processing Name column text data...
Loading SpaCy model (en_core_web_sm)...
Name text processing init completed.
Processing 30000 valid text entries via SpaCy pipe...


SpaCy Text Normalization: 100%|██████████| 30000/30000 [00:19<00:00, 1541.08it/s]


Text processing on name column completed.

Processing Brand column text data...
Brand standardization init completed.


Standardizing Brands: 100%|██████████| 30000/30000 [00:00<00:00, 335686.65it/s]


Brand standardization completed.

Processing Manufacturer column text data...
Manufacturer standardization init completed.


Standardizing Brands: 100%|██████████| 30000/30000 [00:00<00:00, 308580.17it/s]

Manufacturer standardization completed.
Pipeline completed.
📊 Displaying first 5 rows of the cleaned data...


,brand,brand_clean,manufacturer,manufacturer_clean,name,name_clean
0,Universal Music,universal,Universal Music Group / Cash Money,universal,Pink Friday: Roman Reloaded Re-Up (w/dvd),pink friday roman reloaded w dvd
1,Lundberg,lundberg,Lundberg,lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,lundberg organic cinnamon toast rice cakes
2,Lundberg,lundberg,Lundberg,lundberg,Lundberg Organic Cinnamon Toast Rice Cakes,lundberg organic cinnamon toast rice cakes
3,K-Y,k-y,K-Y,k-y,K-Y Love Sensuality Pleasure Gel,k y love sensuality pleasure gel
4,K-Y,k-y,K-Y,k-y,K-Y Love Sensuality Pleasure Gel,k y love sensuality pleasure gel


In [ ]:
print("Starting Category Imputation...")
catg_imputer = CategoryImputer()

# Fit the map on the raw column
catg_imputer.fit_frequency_map(df_cleaned['categories'], display_frequency_map=True)
df_cleaned = catg_imputer.transform_dominant_category(df_cleaned, source_column='categories', target_column='category_by_globle_counter_max')

# Verify Final Counts (Replaces original call to get_value_counts)
print("\n✅ Category Distribution by Global Counter Max:")
DataAnalyzer.analyze_value_distribution(df_cleaned, 'category_by_globle_counter_max')

# Apply First Phrase Imputer
df_cleaned['category_by_first_phrase_in_col'] = df_cleaned['categories'].apply(CategoryImputer.select_first_category)

print("\n✅ Category Distribution by First Phrase:")
DataAnalyzer.analyze_value_distribution(df_cleaned, 'category_by_first_phrase_in_col')

In [ ]:
DataAnalyzer.sample_random_data(df_cleaned, ['name', 'category_by_globle_counter_max', 'category_by_first_phrase_in_col'], 50)

#####  Analysis of categories column
_After comparing results of both category columns with product name_ <br>
**_we desided to go with category_by_globle_counter_max as categories column_**

In [ ]:
null_value_count = df_cleaned['reviews_date'].isna().sum()
print("reviews_date null values count:", null_value_count, "percentage:", (null_value_count / len(df_cleaned)) * 100)

In [ ]:
DataAnalyzer.analyze_value_distribution(df_cleaned, 'reviews_date')

In [ ]:
df_cleaned['reviews_date'].str.strip().str.startswith('hooks').value_counts(dropna=False)

##### Analysis of invalid rows 
**_Removing invalid rows where other column data is moved to date and those rows missing date column values at all_**

In [ ]:
df_cleaned = filter_valid_rows_based_on_date_column(df_cleaned, 'reviews_date')

In [ ]:
if os.path.exists(PROCESSED_DATA_DIR/'df_cleaned.csv'):
    print("Due to spellchecker's performance, we will load the cleaned data from the disk.")
    print("Else Condition take 01hr:52min:00sec to run")
    df_cleaned = pd.read_csv(PROCESSED_DATA_DIR/'df_cleaned.csv')
else:
    text_processor = TextProcessor()
    df_cleaned['reviews_title'].fillna('', inplace=True)
    df_cleaned['reviews_title_cleaned'] = text_processor.process_batch(df_cleaned['reviews_title'], batch_size=1000, nlp_text_cleaner=True)
    df_cleaned['reviews_text_cleaned'] = text_processor.process_batch(df_cleaned['reviews_text'], batch_size=1000, nlp_text_cleaner=True)
    df_valid.to_csv(PROCESSED_DATA_DIR/'df_valid.csv', index=False)

In [ ]:
df_valid = df_cleaned.deepcopy()
df_valid['brand'] = df_valid['brand_clean']
df_valid['manufacturer'] = df_valid['manufacturer_clean']
df_valid['name'] = df_valid['name_clean']
df_valid['categories'] = df_valid['category_by_first_phrase_in_col']
df_valid['reviews_title'] = df_valid['reviews_title_cleaned']
df_valid['reviews_text'] = df_valid['reviews_text_cleaned']
colmns_to_drop = ['brand_clean', 'manufacturer_clean', 'name_clean', 
    'category_by_first_phrase_in_col', 'category_by_globle_counter_max', 
    'reviews_title_cleaned', 'reviews_text_cleaned']
df_valid.drop(columns=colmns_to_drop, inplace= True)

print("Dropping intermediate columns")
print("Final DataFrame shape: ", df_valid.shape)
display(df_valid.head())

In [ ]:
get_missing_columns(df_valid, display_threshold_percent=0.01)

In [ ]:
all_cols = set(df_valid.columns.tolist())
dropna_cols = list(all_cols - set(CONFIG.get_config("dropna_exc_cols")))
df_valid = df_valid.dropna(subset=dropna_cols).reset_index(drop=True)
print("Dropped rows with missing values in the following columns:", dropna_cols)
print("Count:", len(df_raw) - len(df_valid), f"/{len(df_raw)}")
print(f"percentage of dropped rows: {((len(df_raw) - len(df_valid)) * 100/ len(df_raw)):.4f} %")

In [ ]:
get_missing_columns(df_valid, display_threshold_percent=0.01)


#### IMPUTATION of MISSING DATA

In [ ]:
df_valid =  pd.read_csv(PROCESSED_DATA_DIR / "df_valid.csv")

# 4. Univariate Analysis (Distribution Analysis) 📈

# 5. Bivariate & Multivariate Analysis (Feature Relationships) 🤝

# 6. Text-Specific EDA (Natural Language Processing) 💬

# 7. Summary and Key Insights ✨